# Fetch and Map Speeding Events

This notebook uses the KeepTruckin API to fetch speeding events and map them using Folium.

In [ ]:

import requests
import folium
from folium import DivIcon

# Input your KeepTruckin API key
api_key = "API KEY"

# Define the base URL for the KeepTruckin API
base_url = "https://api.keeptruckin.com/v1/speeding_events"

# Create a Folium map with an initial center
m = folium.Map(location=[40.7128, -74.0060], zoom_start=10)

# Initialize summary variables
total_events = 0
severity_counts = {}
first_event_time = None
last_event_time = None

# Function to fetch and map latitude and longitude data
def fetch_and_map_lat_lon(api_key, map):
    global total_events, first_event_time, last_event_time  # Declare global variables
    page_number = 1
    center_set = False

    # Define colors for different severity levels
    severity_colors = {
        "critical": "red",
        "high": "orange",
    }

    # Loop through pages until no events are returned
    while True:
        # Set parameters for API request
        params = {
            "per_page": 100,  # Increase per_page value to fetch more events in each request
            "page_no": page_number,
        }
        headers = {
            "accept": "application/json",
            "X-Api-Key": api_key,
        }

        # Send request to the API
        response = requests.get(base_url, params=params, headers=headers)
        data = response.json()

        # Extract events from API response
        events = data.get("speeding_events", [])

        # If no more events, exit the loop
        if not events:
            break

        # Iterate over the events and add markers to the map
        for event in events:
            # Extract event data
            start_lat = event["speeding_event"]["start_lat"]
            start_lon = event["speeding_event"]["start_lon"]
            end_lat = event["speeding_event"]["end_lat"]
            end_lon = event["speeding_event"]["end_lon"]
            vehicle_name = event["speeding_event"]["vehicle"]["number"]
            event_date = event["speeding_event"]["start_time"]
            severity = event["speeding_event"]["metadata"]["severity"]
            duration_seconds = event["speeding_event"]["duration"]
            event_id = event["speeding_event"]["id"]

            # Update summary values
            total_events += 1
            severity_counts[severity] = severity_counts.get(severity, 0) + 1

            # Update first and last event timestamps
            event_timestamp = event["speeding_event"]["start_time"]
            if first_event_time is None or event_timestamp < first_event_time:
                first_event_time = event_timestamp
            if last_event_time is None or event_timestamp > last_event_time:
                last_event_time = event_timestamp

            # Convert duration from seconds to minutes
            duration_minutes = duration_seconds / 60

            # Create a hyperlink with the specified format
            event_url = f"https://app.gomotive.com/en-US/#/safety/speeding/{event_id}"

            # Create popup content for markers
            popup_content = f"Vehicle: {vehicle_name}<br>Date: {event_date}<br>Severity: {severity}<br>Duration: {duration_minutes:.2f} minutes<br><a href='{event_url}' target='_blank'>Event Details</a>"

            # Determine the marker color based on severity level
            marker_color = severity_colors.get(severity, "yellow")

            # Create custom HTML markers with unique colors and popup content
            folium.Marker(
                location=[start_lat, start_lon],
                icon=DivIcon(
                    icon_size=(20, 20),
                    icon_anchor=(10, 10),
                    html=f'<div style="width: 20px; height: 20px; background-color: {marker_color}; border-radius: 50%;"></div>',
                ),
                popup=popup_content,
            ).add_to(map)

            folium.Marker(
                location=[end_lat, end_lon],
                icon=DivIcon(
                    icon_size=(20, 20),
                    icon_anchor=(10, 10),
                    html=f'<div style="width: 20px; height: 20px; background-color: {marker_color}; border-radius: 50%;"></div>',
                ),
                popup=popup_content,
            ).add_to(map)

            # Set the center of the map based on the first event if not set yet
            if not center_set:
                map.location = [start_lat, start_lon]
                center_set = True

        page_number += 1  # Move to the next page

    # Print summary information
    print(f"Total Events: {total_events}")
    for severity, count in severity_counts.items():
        print(f"Severity {severity.capitalize()}: {count} events")
    print(f"First Event Timestamp: {first_event_time}")
    print(f"Last Event Timestamp: {last_event_time}")
    m

# Fetch and map latitude and longitude data from all pages of speeding events
fetch_and_map_lat_lon(api_key, m)

# Display the map
m
